# KL-IG: Greedy Path Variants — Prototype Evaluation
ResNet50 · Greedy path ablation (SortedDim, GreedyMu, GreedyJoint) vs Linear baseline


# Setup

In [ ]:
!git clone --branch claude/general-session-FcgoB \
    https://github.com/Shameen5375/KLIG_V1.git 2>/dev/null || echo "Repo already cloned"
!pip install -q captum datasets tqdm scikit-learn


In [ ]:
import os, sys, math, json, pickle, warnings
from pathlib import Path
from collections import defaultdict
import copy

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torchvision.models import ResNet50_Weights, resnet50
from scipy import stats
from tqdm.auto import tqdm

ROOT = Path.cwd()
for candidate in [ROOT, ROOT / "infocube-main",
                  Path("/content/KLIG_V1/infocube-main"),
                  Path("/content/KLIG_V1")]:
    if (candidate / "klig").exists():
        ROOT = candidate
        break
sys.path.append(str(ROOT))

from klig.image.attribution import ImageAttributor
from klig.image.stopping import find_sigma_stop
from klig.core.integrator import KLIntegratedGradients
from klig.core.path import LinearPath
from klig.core.greedy_path import SortedDimPath, GreedyMuAttributor, GreedyJointAttributor

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Root:   {ROOT}")


In [ ]:
# ── Sample sizes ──────────────────────────────────────────────────────
N_IMGS    = 1000
N_subset  = 100
N_STEPS   = 50
N_SAMPLES = 10
TIER_A    = 1000

# ── Attribution hyperparameters ────────────────────────────────────────
BLUR_SIGMA  = 16.0
BLUR_KERNEL = 51
IG_STEPS    = 50
EG_SAMPLES  = 50
SG_SAMPLES  = 50
BIG_STEPS   = 50
BIG_SIGMA   = 10.0

# ── Greedy-path hyperparameters ────────────────────────────────────────
GAMMA_LO           = 0.25   # SortedDim: γ for highest-gradient dim
GAMMA_HI           = 4.0    # SortedDim: γ for lowest-gradient dim
SORTED_DIM_SAMPLES = 32     # MC samples for prior gradient estimation
SIGMA_FINAL        = 1 / 256
ADAPTIVE_SIGMA     = True

# ── Metric hyperparameters ─────────────────────────────────────────────
N_INSERTION_STEPS   = 50
N_SENS_SUBSETS      = 30
SENS_FRACTIONS      = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.8]
PERTURBATION_SIGMAS = [0.01, 0.02, 0.05, 0.1, 0.2]
PERTURBATION_RUNS   = 3
OCCLUSION_PATCH     = 14
OCCLUSION_STRIDE    = 7
OCCLUSION_RATIOS    = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]

# ── ImageNet normalisation ─────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
TRANSFORM = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

HF_DATASET_NAME = "evanarlian/imagenet_1k_resized_256"
HF_SPLIT = "val"

# ── Persistent cache ───────────────────────────────────────────────────
USE_DRIVE       = True
DRIVE_CACHE     = "/content/drive/MyDrive/klig_greedy_cache"
LOCAL_CACHE     = "greedy_eval_cache"
FORCE_RECOMPUTE = False
if USE_DRIVE:
    try:
        from google.colab import drive
        if not Path("/content/drive").exists() or not any(Path("/content/drive").iterdir()):
            drive.mount("/content/drive")
        CACHE_DIR = Path(DRIVE_CACHE)
        print(f"[cache] Drive: {CACHE_DIR}")
    except Exception as e:
        CACHE_DIR = Path(LOCAL_CACHE)
        print(f"[cache] Drive unavailable ({e}) — falling back to {CACHE_DIR}")
else:
    CACHE_DIR = Path(LOCAL_CACHE)
    print(f"[cache] Local: {CACHE_DIR}")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Methods + colors ───────────────────────────────────────────────────
methods_all = [
    "KL-IG (adaptive)",
    "KL-IG-SortedDim",
    "KL-IG-GreedyMu",
    "KL-IG-GreedyJoint",
    "KL-IG-Random",
]

COLORS_ALL = {
    "KL-IG (adaptive)":  "#1B5E3F",
    "KL-IG-SortedDim":   "#2196F3",
    "KL-IG-GreedyMu":    "#4CAF50",
    "KL-IG-GreedyJoint": "#FF5722",
    "KL-IG-Random":      "#888888",
}

print(f"N_IMGS={N_IMGS}, methods={len(methods_all)}")
print(f"Cache: {CACHE_DIR.resolve()}")


In [ ]:
def load_model():
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = resnet50(weights=weights).to(DEVICE).eval()
    return model, weights.meta["categories"]

def denormalize(x):
    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(-1, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=x.device).view(-1, 1, 1)
    if x.dim() == 4:
        mean, std = mean.unsqueeze(0), std.unsqueeze(0)
    return (x * std + mean).clamp(0, 1)

def get_sigma_final(model, x, target):
    if ADAPTIVE_SIGMA:
        return min(max(find_sigma_stop(model, x, target=target, tau=0.95), 1.0/256.0), 1.0)
    return SIGMA_FINAL

model, imagenet_labels = load_model()
print(f"Model: ResNet50, {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")


In [ ]:
def load_imagenet_subset(n_images):
    try:
        from datasets import load_dataset
        print(f"[dataset] HuggingFace {HF_DATASET_NAME} [{HF_SPLIT}]")
        ds = load_dataset(HF_DATASET_NAME, split=HF_SPLIT, streaming=True) \
             .shuffle(seed=42, buffer_size=50_000)
        seen, out = set(), []
        for ex in ds:
            y = int(ex["label"])
            if y in seen:
                continue
            seen.add(y)
            out.append((ex["image"].convert("RGB"), y))
            if len(out) >= n_images:
                break
        return out
    except Exception as e:
        print(f"[dataset] Failed: {e}")
        return []

raw_samples = load_imagenet_subset(TIER_A)
dataset = []
with torch.no_grad():
    for i, (pil, gt) in enumerate(tqdm(raw_samples, desc="prep")):
        x = TRANSFORM(pil).unsqueeze(0).to(DEVICE)
        probs = model(x).softmax(-1)[0]
        top1  = int(probs.argmax())
        target = gt if probs[gt].item() > 0.05 else top1
        dataset.append({"idx": i, "x": x, "target": target,
                         "label_str": imagenet_labels[target]})

eval_idx = list(range(min(N_IMGS, len(dataset))))
print(f"Dataset: {len(dataset)} images, evaluating {len(eval_idx)}")


In [ ]:
def absmax_collapse(a):
    if a.dim() == 4: a = a.squeeze(0)
    idx = a.abs().argmax(dim=0, keepdim=True)
    return a.gather(0, idx).squeeze(0)

# attrs stored in all_attrs are already (H, W) — to_heatmap only clips/normalises
def to_heatmap(attr_hw, clip_pct=99.0):
    m = attr_hw.cpu().float()
    clip = float(torch.quantile(m.abs(), clip_pct / 100.0))
    m = m.clamp(-clip, clip)
    lo, hi = m.min(), m.max()
    if hi > lo:
        m = (m - lo) / (hi - lo)
    return m.numpy()


# Attribution loop

In [ ]:
_cache = CACHE_DIR / "greedy_attrs.pkl"

if not FORCE_RECOMPUTE and _cache.exists():
    with open(_cache, "rb") as f:
        all_attrs = pickle.load(f)
    print(f"[cache] Loaded {len(all_attrs)} results from {_cache}")
else:
    all_attrs = defaultdict(dict)   # all_attrs[method][img_idx] = attr (H,W) tensor

    for row in tqdm(dataset, desc="images"):
        idx    = row["idx"]
        x      = row["x"]          # (1, 3, 224, 224)
        target = row["target"]
        sigma  = get_sigma_final(model, x, target)
        x1     = x.squeeze(0)      # (3, 224, 224)

        # ── KL-IG (adaptive, linear path) ──
        ig = KLIntegratedGradients(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=sigma, path=LinearPath(), device=DEVICE
        )
        res = ig.attribute(x1, target=target)
        all_attrs["KL-IG (adaptive)"][idx] = absmax_collapse(res.attr).cpu()

        # ── KL-IG-SortedDim ──
        sp = SortedDimPath.from_model_and_input(
            model, x1, target=target,
            n_samples=SORTED_DIM_SAMPLES,
            gamma_lo=GAMMA_LO, gamma_hi=GAMMA_HI,
        )
        ig_s = KLIntegratedGradients(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=sigma, path=sp, device=DEVICE
        )
        res_s = ig_s.attribute(x1, target=target)
        all_attrs["KL-IG-SortedDim"][idx] = absmax_collapse(res_s.attr).cpu()

        # ── KL-IG-GreedyMu ──
        gmu = GreedyMuAttributor(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=sigma, device=DEVICE
        )
        res_gmu = gmu.attribute(x1, target=target)
        all_attrs["KL-IG-GreedyMu"][idx] = absmax_collapse(res_gmu.attr).cpu()

        # ── KL-IG-GreedyJoint ──
        gjoint = GreedyJointAttributor(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=sigma, device=DEVICE
        )
        res_gj = gjoint.attribute(x1, target=target)
        all_attrs["KL-IG-GreedyJoint"][idx] = absmax_collapse(res_gj.attr).cpu()

        # ── KL-IG-Random ──
        class _RandomPath(LinearPath):
            def steps(self, n):
                return super().steps(n)[torch.randperm(n)]
        ig_r = KLIntegratedGradients(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=sigma, path=_RandomPath(), device=DEVICE
        )
        res_r = ig_r.attribute(x1, target=target)
        all_attrs["KL-IG-Random"][idx] = absmax_collapse(res_r.attr).cpu()

    with open(_cache, "wb") as f:
        pickle.dump(dict(all_attrs), f)
    print(f"[cache] Saved to {_cache}")


# 1. Single-image attribution maps

In [ ]:
# Pick a representative image (first in dataset)
row = dataset[0]
x_vis  = row["x"]          # (1, 3, 224, 224)
target_vis = row["target"]
x_display  = denormalize(x_vis[0]).cpu().permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, len(methods_all) + 1,
                          figsize=(3 * (len(methods_all) + 1), 3.5))
axes[0].imshow(x_display)
axes[0].set_title(f'Input\n{row["label_str"]}', fontsize=9)
axes[0].axis("off")

for ax, m in zip(axes[1:], methods_all):
    attr = all_attrs[m][row["idx"]]
    hm = to_heatmap(attr)
    ax.imshow(hm, cmap="RdBu_r", vmin=0, vmax=1)
    ax.set_title(m, fontsize=8)
    ax.axis("off")

plt.suptitle("Attribution maps — absmax collapse, 99th-pct clip", fontsize=11)
plt.tight_layout()
plt.show()


# 2. Sparsity (Gini coefficient)

In [ ]:
_cache_g = CACHE_DIR / "greedy_gini.pkl"

def gini(v):
    v = v.abs().flatten().sort()[0].float()
    n = len(v)
    idx = torch.arange(1, n + 1, dtype=torch.float)
    return float((2 * (idx * v).sum() / (n * v.sum() + 1e-12) - (n + 1) / n).item())

if not FORCE_RECOMPUTE and _cache_g.exists():
    with open(_cache_g, "rb") as f:
        all_gini = pickle.load(f)
    print(f"[cache] Loaded from {_cache_g}")
else:
    all_gini = {m: [gini(all_attrs[m][i]) for i in eval_idx] for m in methods_all}
    with open(_cache_g, "wb") as f:
        pickle.dump(all_gini, f)

ci95 = lambda v: 1.96 * np.std(v) / (len(v) ** 0.5)
means = [np.mean(all_gini[m]) for m in methods_all]
cis   = [ci95(all_gini[m])   for m in methods_all]

fig, ax = plt.subplots(figsize=(10, 4), facecolor="white")
for xi, (m, mu, ci) in enumerate(zip(methods_all, means, cis)):
    ax.bar(xi, mu, color=COLORS_ALL[m], alpha=0.85, width=0.6)
    ax.errorbar(xi, mu, yerr=ci, fmt="none", color="black", capsize=4, lw=1.5)
ax.set_xticks(range(len(methods_all)))
ax.set_xticklabels(methods_all, rotation=25, ha="right", fontsize=9)
ax.set_ylabel("Gini coefficient (higher = sparser)")
ax.set_title(f"Attribution sparsity — Gini (n={len(eval_idx)} images)")
plt.tight_layout()
plt.show()


# 3. Insertion / Deletion AUC

In [ ]:
_cache_id = CACHE_DIR / "greedy_ins_del.pkl"

def insertion_deletion(model, x, attr_map, target,
                        n_steps=N_INSERTION_STEPS, batch_size=64):
    C, H, W = x.shape[1], x.shape[2], x.shape[3]
    n_pix = H * W
    order = attr_map.detach().view(-1).argsort(descending=True)
    pps = max(1, n_pix // n_steps)
    blur_base = F.avg_pool2d(x, kernel_size=31, stride=1, padding=15)
    ins_scores, del_scores = [], []
    x_ins = blur_base.clone()
    x_del = x.clone()
    with torch.no_grad():
        for step in range(n_steps):
            pixels = order[step * pps: (step + 1) * pps]
            for ch in range(C):
                flat_ins = x_ins[:, ch].reshape(-1)
                flat_del = x_del[:, ch].reshape(-1)
                flat_ins[pixels] = x[:, ch].reshape(-1)[pixels]
                flat_del[pixels] = blur_base[:, ch].reshape(-1)[pixels]
            ins_scores.append(model(x_ins).softmax(-1)[0, target].item())
            del_scores.append(model(x_del).softmax(-1)[0, target].item())
    ins_auc = float(np.trapz(ins_scores) / n_steps)
    del_auc = float(np.trapz(del_scores) / n_steps)
    return ins_auc, del_auc

if not FORCE_RECOMPUTE and _cache_id.exists():
    with open(_cache_id, "rb") as f:
        ins_auc, del_auc = pickle.load(f)
    print(f"[cache] Loaded from {_cache_id}")
else:
    ins_auc = defaultdict(list)
    del_auc = defaultdict(list)
    for row in tqdm(dataset, desc="ins/del"):
        idx = row["idx"]
        x   = row["x"]
        t   = row["target"]
        for m in methods_all:
            attr = all_attrs[m][idx].to(DEVICE).unsqueeze(0)
            i_auc, d_auc = insertion_deletion(model, x, attr, t)
            ins_auc[m].append(i_auc)
            del_auc[m].append(d_auc)
    with open(_cache_id, "wb") as f:
        pickle.dump((dict(ins_auc), dict(del_auc)), f)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor="white")
for ax, (title, aucs) in zip(axes, [("Insertion AUC ↑", ins_auc),
                                      ("Deletion AUC ↓", del_auc)]):
    means_id = [np.mean(aucs[m]) for m in methods_all]
    cis_id   = [1.96 * np.std(aucs[m]) / len(aucs[m]) ** 0.5 for m in methods_all]
    for xi, (m, mu, ci) in enumerate(zip(methods_all, means_id, cis_id)):
        ax.bar(xi, mu, color=COLORS_ALL[m], alpha=0.85, width=0.6)
        ax.errorbar(xi, mu, yerr=ci, fmt="none", color="black", capsize=4, lw=1.5)
    ax.set_xticks(range(len(methods_all)))
    ax.set_xticklabels(methods_all, rotation=25, ha="right", fontsize=9)
    ax.set_title(title)
plt.suptitle(f"Insertion / Deletion AUC (n={len(eval_idx)} images)", fontsize=12)
plt.tight_layout()
plt.show()


# 4. Sensitivity-n

In [ ]:
_cache_sn = CACHE_DIR / "greedy_sens_n.pkl"

def sensitivity_n(model, x, attr_map, target, n_subsets=N_SENS_SUBSETS,
                   fractions=SENS_FRACTIONS):
    C, H, W = x.shape[1], x.shape[2], x.shape[3]
    n_pix  = H * W
    f0     = model(x).softmax(-1)[0, target].item()
    flat_a = attr_map.detach().view(-1).abs().cpu().numpy()
    pccs   = []
    with torch.no_grad():
        for frac in fractions:
            k = max(1, int(n_pix * frac))
            attr_sums, score_drops = [], []
            for _ in range(n_subsets):
                subset = np.random.choice(n_pix, k, replace=False)
                attr_sums.append(flat_a[subset].sum())
                x_mask = x.clone()
                for ch in range(C):
                    x_mask[:, ch].reshape(-1)[subset] = 0.0
                score_drops.append(f0 - model(x_mask).softmax(-1)[0, target].item())
            r, _ = stats.pearsonr(attr_sums, score_drops)
            pccs.append(float(r) if not np.isnan(r) else 0.0)
    return pccs

if not FORCE_RECOMPUTE and _cache_sn.exists():
    with open(_cache_sn, "rb") as f:
        all_sens = pickle.load(f)
    print(f"[cache] Loaded from {_cache_sn}")
else:
    all_sens = defaultdict(lambda: [[] for _ in SENS_FRACTIONS])
    for row in tqdm(dataset, desc="sens-n"):
        idx = row["idx"]
        x   = row["x"]
        t   = row["target"]
        for m in methods_all:
            attr = all_attrs[m][idx].to(DEVICE).unsqueeze(0)
            pccs = sensitivity_n(model, x, attr, t)
            for fi, p in enumerate(pccs):
                all_sens[m][fi].append(p)
    with open(_cache_sn, "wb") as f:
        pickle.dump(dict(all_sens), f)

fig, ax = plt.subplots(figsize=(10, 5), facecolor="white")
fracs = np.array(SENS_FRACTIONS)
for m in methods_all:
    means_sn = [np.mean(all_sens[m][fi]) for fi in range(len(fracs))]
    ax.plot(fracs, means_sn, "-o", color=COLORS_ALL[m], label=m, lw=2)
ax.set_xlabel("Subset fraction n")
ax.set_ylabel("Pearson correlation")
ax.set_title(f"Sensitivity-n PCC (n={len(eval_idx)} images)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# 5. Greedy path trajectories (PCA, single image)

In [ ]:
# Re-run a single image with diagnostic output to visualise trajectories
from sklearn.decomposition import PCA

row_vis = dataset[0]
x1_vis  = row_vis["x"].squeeze(0)
tgt_vis = row_vis["target"]
sigma_vis = get_sigma_final(model, row_vis["x"], tgt_vis)

sp_vis = SortedDimPath.from_model_and_input(
    model, x1_vis, target=tgt_vis,
    n_samples=SORTED_DIM_SAMPLES, gamma_lo=GAMMA_LO, gamma_hi=GAMMA_HI)

gmu_vis = GreedyMuAttributor(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                              sigma_final=sigma_vis, device=DEVICE)
res_gmu_vis = gmu_vis.attribute(x1_vis, target=tgt_vis)

gjoint_vis = GreedyJointAttributor(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                                    sigma_final=sigma_vis, device=DEVICE)
res_gj_vis = gjoint_vis.attribute(x1_vis, target=tgt_vis)

mu_final_np = x1_vis.detach().cpu().reshape(-1).numpy()
ts_np = np.linspace(0.5/N_STEPS, 1.0 - 0.5/N_STEPS, N_STEPS)
gamma_np = sp_vis._gamma.cpu().reshape(-1).numpy()

linear_wps  = np.outer(ts_np, mu_final_np)
sorted_wps  = np.array([ts_np[k]**gamma_np * mu_final_np for k in range(N_STEPS)])
gmu_wps     = torch.stack(res_gmu_vis.waypoints_mu).reshape(N_STEPS, -1).cpu().numpy()
gjoint_wps  = torch.stack(res_gj_vis.waypoints_mu).reshape(N_STEPS, -1).cpu().numpy()

all_wps = np.concatenate([linear_wps, sorted_wps, gmu_wps, gjoint_wps])
pca = PCA(n_components=2).fit(all_wps)

fig, ax = plt.subplots(figsize=(7, 6), facecolor="white")
for label, wps in [
    ("KL-IG (adaptive)",  linear_wps),
    ("KL-IG-SortedDim",   sorted_wps),
    ("KL-IG-GreedyMu",    gmu_wps),
    ("KL-IG-GreedyJoint", gjoint_wps),
]:
    pts = pca.transform(wps)
    ax.plot(pts[:, 0], pts[:, 1], "-o", color=COLORS_ALL[label],
            markersize=3, lw=1.5, label=label, alpha=0.85)

origin = pca.transform(np.zeros((1, len(mu_final_np))))
target_pt = pca.transform(mu_final_np.reshape(1, -1))
ax.scatter(*origin.T, marker="*", s=200, color="black", zorder=5, label="Prior")
ax.scatter(*target_pt.T, marker="D", s=100, color="gold", edgecolors="black",
            zorder=5, label="μ_final")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
ax.set_title("μ-space trajectories: PCA projection")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# 6. Summary table

In [ ]:
import pandas as pd

ci95 = lambda v: 1.96 * np.std(v) / (len(v) ** 0.5)

rows = []
for m in methods_all:
    row = {"Method": m}
    row["Gini ↑"]     = f"{np.mean(all_gini[m]):.4f} ± {ci95(all_gini[m]):.4f}"
    row["Ins AUC ↑"]  = f"{np.mean(ins_auc[m]):.4f} ± {ci95(ins_auc[m]):.4f}"
    row["Del AUC ↓"]  = f"{np.mean(del_auc[m]):.4f} ± {ci95(del_auc[m]):.4f}"
    sn_mean = np.mean([np.mean(all_sens[m][fi]) for fi in range(len(SENS_FRACTIONS))])
    row["Sens-n PCC ↑"] = f"{sn_mean:.4f}"
    rows.append(row)

df = pd.DataFrame(rows).set_index("Method")
print(df.to_string())
